# 第8章 数据库（一）：数据库概念与关系模型
# Chapter 8 Databases (Part 1): Database Concepts & Relational Model

---

**Cambridge International AS & A Level Computer Science (9618)**

> "Bad programmers worry about the code. Good programmers worry about data structures and their relationships."
> — Linus Torvalds

本章我们将深入探索**数据库（Database）**的世界。数据库是现代信息系统的心脏——从你每天使用的微信、淘宝、抖音，到银行系统、医院记录、学校管理，背后都有数据库在默默工作。

**本节学习目标：**
- 理解基于文件的方法（file-based approach）的问题
- 掌握关系数据库（relational database）如何解决这些问题
- 熟练掌握所有数据库术语（entity, record, field, key 等）
- 学会绘制 E-R 图（Entity-Relationship diagram）
- 掌握规范化（normalization）从 UNF 到 3NF 的完整过程

## 8.1.1 基于文件的方法的问题 | Problems with File-Based Approach

### 想象一个没有数据库的世界

在数据库出现之前，人们用**独立的文件（separate files）**来存储数据。想象一所学校：

- 📁 **教务处**有一个文件记录学生的课程成绩
- 📁 **财务处**有一个文件记录学生的缴费情况
- 📁 **图书馆**有一个文件记录学生的借书情况
- 📁 **体育部**有一个文件记录学生的体育成绩

每个部门**独立维护**自己的文件，互不相通。这就是**文件处理方法（file-based approach）**。

听起来似乎还行？让我们看看这会带来什么灾难性的问题……

### 文件处理方法的五大问题

让我们用 Python 来模拟这些问题：

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 SimHei 字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DengXian', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('SimHei')
if 'SimHei' in test_font or 'simhei' in test_font.lower():
    print('✅ 中文字体 SimHei 已加载')
else:
    print(f'⚠️ SimHei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = 'C:/Windows/Fonts/simhei.ttf'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['SimHei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 SimHei 字体文件')

import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: File-based approach (problematic) ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title("File-Based Approach (problematic)", fontsize=14, fontweight='bold', color='red')
ax.axis('off')

departments = [
    ("Academic Office", 1.5, 8, '#ffcccc'),
    ("Finance Office", 1.5, 5.5, '#ccffcc'),
    ("Library", 1.5, 3, '#ccccff'),
    ("Sports Dept", 1.5, 0.5, '#ffffcc'),
]

for name, x, y, color in departments:
    rect = mpatches.FancyBboxPatch((x, y), 3.5, 1.8, boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + 1.75, y + 1.2, name, ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(x + 1.75, y + 0.5, "Name, ID, Phone,\nAddress, ...", ha='center', va='center', fontsize=8, style='italic')

for i in range(len(departments)):
    x1, y1 = 5.0, departments[i][2] + 0.9
    ax.annotate("", xy=(6.5, 5), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5, ls='--'))

ax.text(7, 5, "Same student\ndata repeated\n4 times!", ha='center', va='center',
        fontsize=12, fontweight='bold', color='red',
        bbox=dict(boxstyle='round', facecolor='#ffeeee', edgecolor='red'))

# --- Right: Database approach (solution) ---
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title("Database Approach (solution)", fontsize=14, fontweight='bold', color='green')
ax.axis('off')

db_rect = mpatches.FancyBboxPatch((2.5, 3.5), 5, 3, boxstyle="round,pad=0.2",
                                   facecolor='#e6f3ff', edgecolor='blue', linewidth=3)
ax.add_patch(db_rect)
ax.text(5, 5.5, "Relational Database", ha='center', va='center', fontsize=13, fontweight='bold', color='blue')
ax.text(5, 4.5, "Single Source of Truth", ha='center', va='center', fontsize=11, style='italic', color='darkblue')

apps = [
    ("Academic", 1, 8.5), ("Finance", 4, 8.5),
    ("Library", 7, 8.5), ("Sports", 9, 8.5),
]
for name, x, y in apps:
    rect = mpatches.FancyBboxPatch((x - 0.8, y - 0.5), 1.6, 1, boxstyle="round,pad=0.1",
                                    facecolor='#e8ffe8', edgecolor='green', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, name, ha='center', va='center', fontsize=9, fontweight='bold')
    ax.annotate("", xy=(5, 6.5), xytext=(x, y - 0.5),
                arrowprops=dict(arrowstyle='->', color='green', lw=2))

ax.text(5, 2, "All departments share\none unified database!", ha='center', va='center',
        fontsize=11, fontweight='bold', color='green',
        bbox=dict(boxstyle='round', facecolor='#eeffee', edgecolor='green'))

plt.tight_layout()
plt.savefig("file_vs_db.png", dpi=100, bbox_inches='tight')
plt.show()
print("Left: each department keeps its own copy of student data (redundancy!)")
print("Right: all departments access ONE shared database (no redundancy!)")


### 五大问题详解

#### 1. 数据冗余（Data Redundancy）

**问题：** 相同的数据被存储在多个文件中。

> **现实类比：** 想象你的电话号码被写在10个不同的通讯录里。如果你换了号码，你需要更新10次！

```
教务处文件：张三, 2005001, 13800138001, 北京市朝阳区...
财务处文件：张三, 2005001, 13800138001, 北京市朝阳区...
图书馆文件：张三, 2005001, 13800138001, 北京市朝阳区...
```

同一个学生的姓名、ID、电话、地址被重复存储了3次。这浪费了**存储空间**，更严重的是引发了下面的问题。

#### 2. 数据不一致（Data Inconsistency）

**问题：** 更新了一个文件中的数据，忘记更新其他文件，导致同一个数据在不同地方的值不同。

> **现实类比：** 你搬家后只告诉了快递公司新地址，忘了告诉银行。结果银行的信寄到旧地址——数据不一致！

```
教务处：张三的地址 = 北京市海淀区（已更新 ✓）
财务处：张三的地址 = 北京市朝阳区（忘记更新 ✗）
```

#### 3. 数据孤立（Data Isolation）

**问题：** 数据分散在不同格式的文件中，难以跨部门查询和共享。

> **现实类比：** 你的照片分散在手机、电脑、U盘、云盘里，格式有 JPG、PNG、HEIC……当你想找"去年春节的全家福"时，你得翻遍所有设备！

#### 4. 数据完整性问题（Integrity Problems）

**问题：** 没有统一的规则来确保数据的正确性和有效性。

> **例如：** 学生年龄应该在 5-100 之间，但文件系统无法自动检查，有人可能输入了 -5 或 999。

#### 5. 安全性限制（Security Limitations）

**问题：** 难以对不同用户设置不同的访问权限。

> **例如：** 财务处的人不应该看到学生的体育成绩，但如果数据都在同一个文件里，很难做到精细的权限控制。

In [ ]:
# 用 Python 模拟文件处理方法的问题

# 模拟各部门独立维护的学生文件
academic_file = [
    {"id": "S001", "name": "Zhang San", "phone": "13800138001", "address": "Beijing Haidian", "grade": "A"},
    {"id": "S002", "name": "Li Si",     "phone": "13900139001", "address": "Beijing Chaoyang", "grade": "B"},
]

finance_file = [
    {"id": "S001", "name": "Zhang San", "phone": "13800138001", "address": "Beijing Chaoyang", "fee_paid": True},
    {"id": "S002", "name": "Li Si",     "phone": "13900139002", "address": "Beijing Chaoyang", "fee_paid": False},
]

print("=== Problem 1: Data Redundancy ===")
print("Student Zhang San's name, phone, address stored in BOTH files!")
print(f"Academic file: {academic_file[0]}")
print(f"Finance file:  {finance_file[0]}")

print("\n=== Problem 2: Data Inconsistency ===")
print(f"Academic says Zhang San's address: {academic_file[0]['address']}")
print(f"Finance  says Zhang San's address: {finance_file[0]['address']}")
print("They are DIFFERENT! Which one is correct?")

print("\n=== Problem 3: Data Inconsistency in phone ===")
print(f"Academic says Li Si's phone: {academic_file[1]['phone']}")
print(f"Finance  says Li Si's phone: {finance_file[1]['phone']}")
print("Last digit differs! Data inconsistency!")

## 8.1.2 关系数据库的解决方案 | Relational Database Solution

**关系数据库（Relational Database）**是由 E.F. Codd 在 1970 年提出的革命性概念。

### 核心思想：用"表"组织数据，用"关系"连接表

| 文件处理方法的问题 | 关系数据库的解决方案 |
|:---|:---|
| **数据冗余** Data Redundancy | 每条数据只存储一次（Single Source of Truth） |
| **数据不一致** Data Inconsistency | 只有一份数据，不可能不一致 |
| **数据孤立** Data Isolation | 所有数据在一个系统中，用 SQL 查询 |
| **完整性问题** Integrity Problems | 通过约束（constraints）强制数据合法 |
| **安全限制** Security Limitations | DBMS 提供精细的权限控制 |

> **关键概念 — Single Source of Truth（单一事实来源）：**
> 每条数据只在数据库中存储一次。需要引用这条数据的地方，都通过**外键（Foreign Key）**指向同一条记录。这样，更新只需一次，所有引用自动获得最新值。

## 8.1.3 数据库术语大全 | Database Terminology

这是考试中**必须精确掌握**的术语。让我们逐一深入理解：

### 基本结构术语

| 术语 (Term) | 含义 (Meaning) | 类比 (Analogy) |
|:---|:---|:---|
| **Entity** 实体 | 数据库中要存储信息的对象（如学生、课程） | 一类"东西" |
| **Table / Relation** 表/关系 | 存储某类实体数据的二维表格 | Excel 中的一个工作表 |
| **Record / Tuple** 记录/元组 | 表中的一行，代表一个具体的实体实例 | 表格中的一行数据 |
| **Field / Attribute** 字段/属性 | 表中的一列，代表实体的某个特征 | 表格中的一个列标题 |

### 关于 Key（键）—— 数据库的"身份证系统"

| Key 类型 | 定义 | 例子 |
|:---|:---|:---|
| **Primary Key** 主键 | 唯一标识表中每条记录的字段（或字段组合） | 学号 StudentID |
| **Candidate Key** 候选键 | 任何能唯一标识记录的字段，都是候选键 | 学号、身份证号都可以 |
| **Secondary Key** 辅助键 | 用于搜索和排序的字段（不一定唯一） | 按姓名搜索学生 |
| **Foreign Key** 外键 | 一个表中引用另一个表主键的字段 | 成绩表中的 StudentID |

### 为什么需要 Primary Key？

> **思考：** 如果一个班有两个叫"张三"的学生，你怎么区分他们？
>
> 这就是为什么需要主键——一个**绝对唯一**的标识符，确保每条记录都能被精确找到。就像每个人的身份证号是唯一的一样。

### 什么是 Referential Integrity（参照完整性）？

> 当表A的外键引用表B的主键时，外键的值必须在表B中存在（或为 NULL）。
>
> **例如：** 成绩表中的 StudentID 必须对应一个真实存在的学生。你不能给一个不存在的学生录入成绩！

In [ ]:
# 用 Python 创建示例表，展示所有术语

print("=" * 70)
print("STUDENTS Table (学生表) — Entity: Student")
print("=" * 70)

# Define the table
students = [
    {"StudentID": "S001", "Name": "Zhang San", "Age": 17, "ClassID": "C01"},
    {"StudentID": "S002", "Name": "Li Si",     "Age": 16, "ClassID": "C01"},
    {"StudentID": "S003", "Name": "Wang Wu",   "Age": 17, "ClassID": "C02"},
    {"StudentID": "S004", "Name": "Zhao Liu",  "Age": 16, "ClassID": "C02"},
]

# Print as table
header = f"{'StudentID':<12} {'Name':<12} {'Age':<6} {'ClassID':<8}"
print(header)
print("-" * len(header))
for s in students:
    print(f"{s['StudentID']:<12} {s['Name']:<12} {s['Age']:<6} {s['ClassID']:<8}")

print()
print("Terminology breakdown:")
print(f"  Entity      = Student (the 'thing' we are storing)")
print(f"  Table       = The entire STUDENTS table above")
print(f"  Record/Tuple = Each row, e.g. (S001, Zhang San, 17, C01)")
print(f"  Field/Attr  = Each column: StudentID, Name, Age, ClassID")
print(f"  Primary Key = StudentID (uniquely identifies each student)")
print(f"  Foreign Key = ClassID (references the CLASSES table)")

print("\n" + "=" * 70)
print("CLASSES Table (班级表)")
print("=" * 70)
classes = [
    {"ClassID": "C01", "ClassName": "Year 12 Science", "TeacherID": "T01"},
    {"ClassID": "C02", "ClassName": "Year 12 Arts",    "TeacherID": "T02"},
]
header2 = f"{'ClassID':<10} {'ClassName':<20} {'TeacherID':<10}"
print(header2)
print("-" * len(header2))
for c in classes:
    print(f"{c['ClassID']:<10} {c['ClassName']:<20} {c['TeacherID']:<10}")

print()
print("Here ClassID is the PRIMARY KEY of CLASSES table.")
print("In STUDENTS table, ClassID is a FOREIGN KEY referencing CLASSES.ClassID")
print("This LINKS the two tables together!")

### Candidate Key vs Primary Key 的区别

一个表可以有多个 **candidate key**（候选键），但只能选一个作为 **primary key**（主键）。

**例子：** 在学生表中：
- `StudentID`（学号）可以唯一标识学生 → 候选键 ✓
- `IDCardNumber`（身份证号）也可以唯一标识学生 → 候选键 ✓
- `Name`（姓名）不能唯一标识（可能重名）→ 不是候选键 ✗

我们从候选键中选择 `StudentID` 作为主键（因为它更短、更方便）。

### Secondary Key（辅助键）与 Indexing（索引）

**Secondary key** 是用来**搜索和排序**的字段。它不需要唯一。

> **类比：** 想象一本600页的教科书。
> - **Primary Key** 就像页码——每一页有唯一的页码，你可以精确定位。
> - **Index（索引）** 就像书后面的"索引"——按关键词字母排序，告诉你"数据库"出现在第 52、78、134 页。
> - **Secondary Key** 就是索引中的关键词——不唯一，但帮你快速找到数据。

**为什么索引能加速搜索？**

没有索引时，数据库必须**逐行扫描**整个表（就像翻遍整本书找一个词）。
有了索引后，数据库可以用类似**二分查找**的方法快速定位（就像查目录直接翻到对应页）。

索引的代价：占用额外存储空间，且插入/更新/删除时需要同步更新索引。

### 关系类型 | Relationship Types

表与表之间通过外键建立**关系（Relationship）**。关系有三种类型：

#### 1. 一对一关系（One-to-One, 1:1）

每个实体A最多对应一个实体B，反之亦然。

**例子：** 每个人只有一个护照，每本护照只属于一个人。
```
Person 1 ──── 1 Passport
```

#### 2. 一对多关系（One-to-Many, 1:M）

一个实体A可以对应多个实体B，但每个B只对应一个A。

**例子：** 一个班级有多个学生，但每个学生只属于一个班级。
```
Class 1 ──── M Student
```
> 这是最常见的关系类型！

#### 3. 多对多关系（Many-to-Many, M:N）

一个实体A可以对应多个实体B，反之亦然。

**例子：** 一个学生可以选多门课程，一门课程可以有多个学生。
```
Student M ──── N Course
```

> **重要：** 在关系数据库中，多对多关系不能直接实现！必须通过一个**中间表（junction table / link table）**拆分为两个一对多关系。
>
> ```
> Student 1 ──── M Enrollment M ──── 1 Course
> ```

## 8.1.4 实体关系图 | Entity-Relationship (E-R) Diagrams

E-R 图是设计数据库的蓝图，由 Peter Chen 在 1976 年提出。

### E-R 图的基本元素：
- **矩形（Rectangle）** → 实体（Entity）
- **椭圆（Ellipse）** → 属性（Attribute）
- **菱形（Diamond）** → 关系（Relationship）
- **线（Line）** → 连接实体与关系，标注基数（1, M, N）

让我们画一个学校数据库的 E-R 图：

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title("E-R Diagram: School Database", fontsize=16, fontweight='bold')

def draw_entity(ax, x, y, label, color='#d4e6f1'):
    rect = mpatches.FancyBboxPatch((x - 1.2, y - 0.5), 2.4, 1.0,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=12, fontweight='bold')

def draw_relationship(ax, x, y, label, color='#fadbd8'):
    diamond = plt.Polygon([(x, y+0.6), (x+1.2, y), (x, y-0.6), (x-1.2, y)],
                          facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(diamond)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

def draw_attribute(ax, x, y, label, underline=False):
    ellipse = mpatches.Ellipse((x, y), 2.0, 0.7, facecolor='#fef9e7', edgecolor='black', linewidth=1)
    ax.add_patch(ellipse)
    style = 'normal'
    if underline:
        ax.text(x, y, label, ha='center', va='center', fontsize=9,
                fontweight='bold', style=style,
                bbox=dict(boxstyle='round,pad=0.05', facecolor='none', edgecolor='none'))
        ax.plot([x - len(label)*0.12, x + len(label)*0.12], [y - 0.12, y - 0.12], 'k-', lw=1.5)
    else:
        ax.text(x, y, label, ha='center', va='center', fontsize=9)

# Entities
draw_entity(ax, 3, 5, "STUDENT", '#aed6f1')
draw_entity(ax, 13, 5, "COURSE", '#a9dfbf')
draw_entity(ax, 3, 1.5, "TEACHER", '#f9e79f')

# Relationships
draw_relationship(ax, 8, 5, "Enrolls\nIn")
draw_relationship(ax, 8, 1.5, "Teaches")

# Lines with cardinality
ax.plot([4.2, 6.8], [5, 5], 'k-', lw=2)
ax.text(4.8, 5.3, "M", fontsize=14, fontweight='bold', color='red')
ax.plot([9.2, 11.8], [5, 5], 'k-', lw=2)
ax.text(11.2, 5.3, "N", fontsize=14, fontweight='bold', color='red')

ax.plot([3, 3], [4.5, 2.1], 'k-', lw=2)
ax.text(3.3, 2.5, "M", fontsize=14, fontweight='bold', color='red')
ax.plot([4.2, 6.8], [1.5, 1.5], 'k-', lw=2)

ax.plot([9.2, 13], [1.5, 4.5], 'k-', lw=2)
ax.text(6.5, 1.8, "1", fontsize=14, fontweight='bold', color='blue')
ax.text(12.2, 2.8, "M", fontsize=14, fontweight='bold', color='red')

# Student attributes
for attr, pos, ul in [("StudentID", (1, 7), True), ("Name", (3, 7.5), False),
                       ("Age", (5, 7), False), ("Email", (3, 8.2), False)]:
    draw_attribute(ax, pos[0], pos[1], attr, ul)
    ax.plot([pos[0], 3], [pos[1] - 0.35, 5.5], 'k-', lw=0.8)

# Course attributes
for attr, pos, ul in [("CourseID", (11, 7), True), ("CourseName", (13, 7.5), False),
                       ("Credits", (15, 7), False)]:
    draw_attribute(ax, pos[0], pos[1], attr, ul)
    ax.plot([pos[0], 13], [pos[1] - 0.35, 5.5], 'k-', lw=0.8)

# Teacher attributes
for attr, pos, ul in [("TeacherID", (1, 0.2), True), ("TName", (5.5, 0.2), False)]:
    draw_attribute(ax, pos[0], pos[1], attr, ul)
    ax.plot([pos[0], 3], [pos[1] + 0.35, 1.0], 'k-', lw=0.8)

# Legend
ax.text(0.5, 9.5, "Legend:", fontsize=11, fontweight='bold')
ax.text(0.5, 9.0, "Rectangle = Entity  |  Diamond = Relationship  |  Ellipse = Attribute", fontsize=9)
ax.text(0.5, 8.6, "Underlined attribute = Primary Key  |  M/N = Many  |  1 = One", fontsize=9)

plt.tight_layout()
plt.savefig("er_diagram.png", dpi=100, bbox_inches='tight')
plt.show()
print("E-R Diagram showing:")
print("  STUDENT M:N COURSE (many-to-many via Enrolls In)")
print("  TEACHER 1:M COURSE (one teacher teaches many courses)")
print("  STUDENT M:1 TEACHER (many students have one teacher, shown via class)")

## 8.1.5 规范化 | Normalization

### 为什么要规范化？

规范化是将数据库设计得**更高效、更可靠**的过程。它的目标是：

1. **消除数据冗余**（Eliminate redundancy）
2. **防止更新异常**（Prevent update anomalies）
3. **防止插入异常**（Prevent insertion anomalies）
4. **防止删除异常**（Prevent deletion anomalies）

### 三大异常（Anomalies）详解

让我们用一个**未规范化（Unnormalized）**的表来理解：

| OrderID | CustomerName | CustomerPhone | Product1 | Price1 | Product2 | Price2 |
|:---|:---|:---|:---|:---|:---|:---|
| 001 | Zhang San | 138xxx | Laptop | 5999 | Mouse | 99 |
| 002 | Li Si | 139xxx | Keyboard | 299 | | |
| 003 | Zhang San | 138xxx | Phone | 3999 | Case | 49 |

#### 更新异常（Update Anomaly）
如果 Zhang San 换了电话号码，我们需要更新**所有包含他信息的行**。如果漏了一行，就产生了不一致。

#### 插入异常（Insertion Anomaly）
如果有一个新客户注册了但还没下单，我们无法插入他的信息（因为没有 OrderID）。

#### 删除异常（Deletion Anomaly）
如果删除订单 002，Li Si 的客户信息也会一起丢失！

In [ ]:
# Normalization process visualization

print('=' * 80)
print('STEP 0: Unnormalized Form (UNF) - The Messy Starting Point')
print('=' * 80)
print()

unf_data = ('| OrderID | Customer | Phone    | Items                          |\n'
            '|---------|----------|----------|--------------------------------|\n'
            '| 001     | Zhang San| 138xxx   | Laptop:5999, Mouse:99          |\n'
            '| 002     | Li Si    | 139xxx   | Keyboard:299                   |\n'
            '| 003     | Zhang San| 138xxx   | Phone:3999, Case:49            |')
print(unf_data)
print()
print('Problem: Items column contains MULTIPLE values (repeating groups)!')
print('This violates the rules of a proper table.')

print()
print('=' * 80)
print('STEP 1: First Normal Form (1NF)')
print('Rule: No repeating groups. Each cell contains ONE atomic value.')
print('=' * 80)
print()

nf1_data = ('| OrderID | Customer  | Phone  | Product  | Price |\n'
            '|---------|-----------|--------|----------|-------|\n'
            '| 001     | Zhang San | 138xxx | Laptop   | 5999  |\n'
            '| 001     | Zhang San | 138xxx | Mouse    | 99    |\n'
            '| 002     | Li Si     | 139xxx | Keyboard | 299   |\n'
            '| 003     | Zhang San | 138xxx | Phone    | 3999  |\n'
            '| 003     | Zhang San | 138xxx | Case     | 49    |')
print(nf1_data)
print()
print('Now each cell has ONE value. Composite Primary Key: (OrderID, Product)')
print('But look: Zhang San name and phone appear 3 times (redundancy)!')
print('Customer/Phone depend only on OrderID, not on the full key (OrderID, Product)')
print('This is called a PARTIAL DEPENDENCY.')

In [ ]:
print('=' * 80)
print('STEP 2: Second Normal Form (2NF)')
print('Rule: In 1NF + No partial dependencies')
print('  (All non-key fields depend on the WHOLE primary key)')
print('=' * 80)
print()
print('We split the table to remove partial dependencies:')
print()

print('Table 1: ORDERS')
print('-' * 45)
orders_2nf = ('| OrderID (PK) | Customer  | Phone  |\n'
              '|--------------|-----------|--------|\n'
              '| 001          | Zhang San | 138xxx |\n'
              '| 002          | Li Si     | 139xxx |\n'
              '| 003          | Zhang San | 138xxx |')
print(orders_2nf)

print()
print('Table 2: ORDER_ITEMS')
print('-' * 45)
items_2nf = ('| OrderID (FK) | Product (PK) | Price |\n'
             '|--------------|--------------|-------|\n'
             '| 001          | Laptop       | 5999  |\n'
             '| 001          | Mouse        | 99    |\n'
             '| 002          | Keyboard     | 299   |\n'
             '| 003          | Phone        | 3999  |\n'
             '| 003          | Case         | 49    |')
print(items_2nf)
print()
print('Now Customer and Phone depend on OrderID alone (no partial dependency).')
print('Product and Price are in their own table with composite key (OrderID, Product).')
print()
print('But wait - Zhang San info is STILL repeated in ORDERS (rows 001 and 003)!')
print('Customer and Phone depend on OrderID, but they REALLY depend on Customer.')
print('Phone is determined by Customer, not directly by OrderID.')
print('This is a TRANSITIVE DEPENDENCY: OrderID -> Customer -> Phone')

In [ ]:
print('=' * 80)
print('STEP 3: Third Normal Form (3NF)')
print('Rule: In 2NF + No transitive dependencies')
print('  (Non-key fields depend ONLY on the primary key, nothing else)')
print('=' * 80)
print()
print('We split again to remove transitive dependencies:')
print()

print('Table 1: CUSTOMERS')
print('-' * 45)
cust_tbl = ('| CustomerID (PK) | CustomerName | Phone  |\n'
            '|-----------------|--------------|--------|\n'
            '| CU01            | Zhang San    | 138xxx |\n'
            '| CU02            | Li Si        | 139xxx |')
print(cust_tbl)

print()
print('Table 2: ORDERS')
print('-' * 45)
ord_tbl = ('| OrderID (PK) | CustomerID (FK) |\n'
           '|--------------|-----------------|\n'
           '| 001          | CU01            |\n'
           '| 002          | CU02            |\n'
           '| 003          | CU01            |')
print(ord_tbl)

print()
print('Table 3: ORDER_ITEMS')
print('-' * 45)
items_tbl = ('| OrderID (FK) | Product  | Price |\n'
             '|--------------|----------|-------|\n'
             '| 001          | Laptop   | 5999  |\n'
             '| 001          | Mouse    | 99    |\n'
             '| 002          | Keyboard | 299   |\n'
             '| 003          | Phone    | 3999  |\n'
             '| 003          | Case     | 49    |')
print(items_tbl)

print()
print('Now:')
print('  - Each customer info stored ONCE (no redundancy)')
print('  - Update Zhang San phone: change ONE row in CUSTOMERS')
print('  - Can add a customer without an order (no insertion anomaly)')
print('  - Deleting order 002 does NOT lose Li Si info (no deletion anomaly)')
print()
print('This is Third Normal Form (3NF)!')

### 规范化总结

| 范式 | 规则 | 消除的问题 |
|:---|:---|:---|
| **1NF** | 每个字段只包含原子值（atomic value），没有重复组（repeating group） | 多值字段 |
| **2NF** | 满足 1NF + 没有部分依赖（partial dependency），即所有非键字段都依赖于**整个**主键 | 部分依赖导致的冗余 |
| **3NF** | 满足 2NF + 没有传递依赖（transitive dependency），即非键字段之间没有依赖关系 | 传递依赖导致的冗余 |

> **记忆口诀：**
> - 1NF: "The key" — 必须有主键，每个字段是原子的
> - 2NF: "The whole key" — 非键字段依赖于整个主键
> - 3NF: "Nothing but the key" — 非键字段只依赖于主键，不依赖于其他非键字段
>
> 完整口诀：**"The key, the whole key, and nothing but the key, so help me Codd!"**
> （致敬关系数据库之父 E.F. Codd）

### 动手练习：规范化实践 | Interactive Normalization Exercise

下面我们用 Python 实际演示一个完整的规范化过程：

In [ ]:
# Interactive Normalization Exercise

print("=" * 70)
print("SCENARIO: A school wants to store student course enrollment data")
print("=" * 70)
print()

# UNF - the raw messy data
print("UNNORMALIZED DATA collected from paper forms:")
print("-" * 70)
unnormalized = [
    {"StudentID": "S01", "StudentName": "Alice", "Courses": "Math, Physics", "Teachers": "Mr.Wang, Ms.Li", "Grades": "A, B+"},
    {"StudentID": "S02", "StudentName": "Bob",   "Courses": "Math, English", "Teachers": "Mr.Wang, Ms.Chen", "Grades": "B, A"},
    {"StudentID": "S03", "StudentName": "Carol", "Courses": "Physics",       "Teachers": "Ms.Li",            "Grades": "A+"},
]
for row in unnormalized:
    print(row)

print()
print("Step 1 -> 1NF: Remove repeating groups (one value per cell)")
print("-" * 70)
nf1 = [
    ("S01", "Alice", "Math",    "Mr.Wang", "A"),
    ("S01", "Alice", "Physics", "Ms.Li",   "B+"),
    ("S02", "Bob",   "Math",    "Mr.Wang", "B"),
    ("S02", "Bob",   "English", "Ms.Chen", "A"),
    ("S03", "Carol", "Physics", "Ms.Li",   "A+"),
]
print(f"{'StuID':<8}{'Name':<8}{'Course':<10}{'Teacher':<10}{'Grade':<6}")
print("-" * 42)
for row in nf1:
    print(f"{row[0]:<8}{row[1]:<8}{row[2]:<10}{row[3]:<10}{row[4]:<6}")
print("Composite PK: (StudentID, Course)")
print("Partial dep: StudentName depends only on StudentID, NOT on Course")
print("Partial dep: Teacher depends only on Course, NOT on StudentID")

print()
print("Step 2 -> 2NF: Remove partial dependencies")
print("-" * 70)
print()
print("STUDENTS table:  StudentID(PK) | StudentName")
students_2 = [("S01", "Alice"), ("S02", "Bob"), ("S03", "Carol")]
for s in students_2:
    print(f"  {s[0]:<12}{s[1]}")

print()
print("COURSES table:   Course(PK) | Teacher")
courses_2 = [("Math", "Mr.Wang"), ("Physics", "Ms.Li"), ("English", "Ms.Chen")]
for c in courses_2:
    print(f"  {c[0]:<12}{c[1]}")

print()
print("ENROLLMENTS:     StudentID(FK) | Course(FK) | Grade")
enrollments = [("S01","Math","A"), ("S01","Physics","B+"), ("S02","Math","B"),
               ("S02","English","A"), ("S03","Physics","A+")]
for e in enrollments:
    print(f"  {e[0]:<15}{e[1]:<12}{e[2]}")

print()
print("Step 3 -> 3NF: Check for transitive dependencies")
print("-" * 70)
print("In STUDENTS: StudentName depends only on StudentID. OK!")
print("In COURSES: Teacher depends only on Course. OK!")
print("In ENROLLMENTS: Grade depends on (StudentID, Course). OK!")
print()
print("Already in 3NF! No transitive dependencies found.")
print("Final design has 3 clean tables with no redundancy.")

## 8.1.6 索引（Indexing）

索引是数据库性能优化的关键技术。

### 索引的工作原理

> **类比：** 一本500页的书，没有目录时你只能从头翻到尾。有了目录（索引），你可以直接翻到对应页码。
>
> 数据库索引的原理类似：它创建一个**排序的数据结构**（通常是 B-tree），存储字段值和对应记录的位置指针。

### 索引的优缺点

| 优点 | 缺点 |
|:---|:---|
| 大幅加速 SELECT 查询 | 占用额外存储空间 |
| 加速 WHERE 条件过滤 | INSERT/UPDATE/DELETE 变慢（需更新索引） |
| 加速 ORDER BY 排序 | 需要维护成本 |

**最佳实践：** 对经常用于搜索的字段（secondary key）建立索引，但不要对很少查询的字段建索引。

In [ ]:
# 模拟索引加速搜索的效果
import time
import random

# Create a "database" with 100000 records
random.seed(42)
database = [{"id": i, "name": f"Student_{i}", "score": random.randint(0, 100)} for i in range(100000)]

# Search WITHOUT index: linear scan
target_score = 95
start = time.perf_counter()
results_no_index = [r for r in database if r["score"] == target_score]
time_no_index = time.perf_counter() - start

# Build an "index" on score field (like a dictionary/hash map)
score_index = {}
for record in database:
    s = record["score"]
    if s not in score_index:
        score_index[s] = []
    score_index[s].append(record)

# Search WITH index: direct lookup
start = time.perf_counter()
results_with_index = score_index.get(target_score, [])
time_with_index = time.perf_counter() - start

print(f"Database size: {len(database):,} records")
print(f"Searching for score = {target_score}")
print(f"  Without index: {time_no_index*1000:.4f} ms, found {len(results_no_index)} records")
print(f"  With index:    {time_with_index*1000:.4f} ms, found {len(results_with_index)} records")
print(f"  Speedup:       ~{time_no_index/max(time_with_index, 0.0000001):.0f}x faster!")
print()
print("This demonstrates why indexing is so important for large databases.")
print("The index trades storage space for dramatically faster lookups.")

## 练习题 | Practice Exercises

### 练习 1：术语匹配
将以下定义与正确的术语配对：

| 定义 | 选项 |
|:---|:---|
| 唯一标识表中每条记录的字段 | A. Foreign Key |
| 引用另一个表主键的字段 | B. Entity |
| 数据库中要存储信息的对象类型 | C. Primary Key |
| 表中的一行数据 | D. Attribute |
| 表中的一列 | E. Record/Tuple |

### 练习 2：识别异常
以下未规范化的表存在什么异常？

| EmpID | EmpName | DeptID | DeptName | DeptManager |
|:---|:---|:---|:---|:---|
| E01 | Alice | D1 | Sales | Mr. Zhang |
| E02 | Bob | D1 | Sales | Mr. Zhang |
| E03 | Carol | D2 | IT | Ms. Li |

思考：
1. 如果 Mr. Zhang 不再管理 Sales 部门，需要更新几条记录？（更新异常）
2. 如果想添加一个新部门 D3-Marketing（还没有员工），能添加吗？（插入异常）
3. 如果删除 Carol 的记录，IT 部门的信息会怎样？（删除异常）

### 练习 3：规范化
将上面的员工-部门表规范化到 3NF。写出最终的表结构。

### 练习 4：E-R 图
画出以下场景的 E-R 图：
- 一个图书馆系统有**读者（Reader）**和**图书（Book）**
- 一个读者可以借多本书，一本书同一时间只能被一个读者借
- 这是什么关系类型？

In [ ]:
# Practice Exercise Answers

print("=" * 60)
print("Exercise 1 Answers:")
print("=" * 60)
print("Primary Key         -> C")
print("Foreign Key          -> A")
print("Entity               -> B")
print("Record/Tuple         -> E")
print("Attribute            -> D")

print()
print("=" * 60)
print("Exercise 2 Answers:")
print("=" * 60)
print("1. Update anomaly: need to update 2 rows (E01 and E02)")
print("   if Mr. Zhang is replaced as Sales manager.")
print("2. Insertion anomaly: cannot add D3-Marketing without")
print("   an employee (EmpID is needed for primary key).")
print("3. Deletion anomaly: deleting Carol loses ALL info")
print("   about the IT department!")

print()
print("=" * 60)
print("Exercise 3 Answer - Normalized to 3NF:")
print("=" * 60)
print()
print("DEPARTMENTS table:")
print("  DeptID (PK) | DeptName   | DeptManager")
print("  D1          | Sales      | Mr. Zhang")
print("  D2          | IT         | Ms. Li")
print()
print("EMPLOYEES table:")
print("  EmpID (PK) | EmpName | DeptID (FK)")
print("  E01        | Alice   | D1")
print("  E02        | Bob     | D1")
print("  E03        | Carol   | D2")
print()
print("Now each department's info is stored ONCE!")

print()
print("=" * 60)
print("Exercise 4 Answer:")
print("=" * 60)
print("Reader 1 --- M Book  (One-to-Many)")
print("One reader can borrow many books,")
print("but each book is borrowed by at most one reader at a time.")